# Day3 · 블록1 · 파일 I/O · JSON · 정규표현식 실습

> **강의자료**: `강의자료/03.01.Python-File-JSON.md`

| Part | 주제 |
|------|------|
| Part 1 | 파일 I/O + JSON 처리 |
| Part 2 | 정규표현식 (Regex) |
| Part 3 | 퀴즈 · 복습 문제 |

로봇 소프트웨어에서 센서 로그 저장, 설정 파일 관리, 비정형 로그 파싱에
필수적인 파일 I/O, JSON 처리, 정규표현식을 실습합니다.

---
## Part 1: 파일 I/O + JSON 처리

### 1.1 로봇 개발에서 파일 I/O가 필요한 이유

로봇은 실시간 동작뿐 아니라 상태를 기록하고 외부 설정을 읽어오는 기능이 반드시 필요합니다.

- **센서 로그 저장**: IMU, 엔코더, 배터리 잔량 등의 텔레메트리 데이터를 저장하여 성능 분석 및 하드웨어 결함 진단에 활용
- **설정 파일 (Configuration)**: 센서 보정값(Calibration offset), 네트워크 IP 주소를 외부 파일에 저장해 코드 수정 없이 동작 환경 변경 가능
- **지도 데이터**: 자율 주행 로봇이 사용하는 점유 격자 지도(Occupancy Grid) 저장·불러오기

### 1.2 open() 함수 — 파일 열기 모드

파이썬에서 파일을 다루는 가장 기본적인 함수: `open()`

| 모드 | 설명 |
|------|------|
| `'r'` (Read) | 읽기 전용, 파일이 없으면 오류 발생 |
| `'w'` (Write) | 쓰기 전용, 기존 내용 삭제 후 새로 작성, 없으면 신규 생성 |
| `'a'` (Append) | 파일 끝에 내용 추가, 기존 내용 유지 |
| `'rb'`, `'wb'` (Binary) | 이미지·센서 원시 데이터(Raw data) 처리 |

In [1]:
# 기본 사용 예시 (권장되지 않음: close()를 직접 호출해야 함)
f = open("sensor_log.txt", "w")
f.write("Sensor Data: 10.5")
f.close()  # 직접 닫아줘야 함 → 실수하기 쉬움

print("파일 작성 완료 (close() 직접 호출)")

파일 작성 완료 (close() 직접 호출)


### 1.3 with 문을 써야 하는 이유 — 파일 핸들 누수 방지

파일을 열고 나면 반드시 `close()`를 호출해야 하지만, 예외 발생 시 잊기 쉽습니다.

- **파일 핸들 누수 방지**: 파일을 닫지 않으면 시스템 자원이 계속 점유되어 다른 프로그램의 파일 접근 차단 및 시스템 성능 저하 가능
- **`with` 문: 자동 종료 보장**: 블록이 끝나는 즉시 파이썬이 자동으로 파일을 안전하게 닫아주며, 데이터 처리 도중 오류가 발생해도 파일을 확실히 닫음

In [2]:
# 권장 방식: with 문 사용
with open("sensor_log.txt", "w", encoding="utf-8") as f:
    f.write("Sensor Data: 10.5")
# 블록을 벗어나는 순간 자동으로 f.close() 호출

### 1.4 JSON이 로봇 개발에서 자주 쓰이는 이유

**JSON (JavaScript Object Notation)**: 가볍고 사람이 읽기 쉬운 데이터 형식

- **가독성**: 텍스트 형식이므로 메모장으로 직접 열어 설정값 수정 가능
- **파이썬과의 호환성**: JSON 구조 ↔ 파이썬의 딕셔너리(Dictionary) 및 리스트(List)와 자연스럽게 매핑
- **확장성**: 동일 내비게이션 SW에서 JSON 파일만 교체하면 다른 로봇에 바로 적용

```json
{
    "robot_id": "Alpha-01",
    "wheel_diameter": 0.1,
    "sensors": ["lidar", "imu"]
}
```

### 1.5 JSON 처리 4가지 핵심 함수

파이썬 `json` 모듈의 핵심 함수 4가지

| 함수 | 대상 | 설명 |
| :--- | :--- | :--- |
| **`json.dump()`** | **파일** | 파이썬 객체 → JSON 형식의 **파일**로 저장 |
| **`json.load()`** | **파일** | JSON 형식의 **파일** → 파이썬 객체로 변환 |
| **`json.dumps()`** | **문자열** | 파이썬 객체 → JSON 형식의 **문자열**로 변환 |
| **`json.loads()`** | **문자열** | JSON 형식의 **문자열** → 파이썬 객체로 변환 |

**구분 핵심:** 끝에 `s`가 붙으면 **String(문자열)** 대상
- `dump` / `load` → 파일 객체
- `dumps` / `loads` → 문자열

### 1.6 센서 로그 데이터를 JSON으로 저장하고 불러오기

In [3]:
import json
import datetime

# 1. 저장할 센서 데이터 준비
sensor_data = {
    "timestamp": str(datetime.datetime.now()),
    "robot_id": "Alpha-01",
    "lidar_samples": [1.2, 1.5, 0.8, 2.3, 1.1],
    "battery_level": 85.5,
    "status": "NORMAL"
}

In [4]:
# 2. JSON 파일로 저장하기 (json.dump)
try:
    with open("robot_logs.json", "w", encoding="utf-8") as f:
        # indent=4: 사람이 읽기 좋게 들여쓰기
        json.dump(sensor_data, f, indent=4)
    print("로그 저장 완료!")
except Exception as e:
    print(f"저장 중 오류 발생: {e}")

로그 저장 완료!


In [5]:
# 3. JSON 파일 불러오기 (json.load)
try:
    with open("robot_logs.json", "r", encoding="utf-8") as f:
        loaded_data = json.load(f)
        print(f"불러온 로봇 ID: {loaded_data['robot_id']}")
        print(f"배터리 상태: {loaded_data['battery_level']}%")
except FileNotFoundError:
    print("파일을 찾을 수 없습니다.")

불러온 로봇 ID: Alpha-01
배터리 상태: 85.5%


### 1.7 자주 발생하는 파일 I/O 오류와 해결법

로봇 소프트웨어는 하드웨어와 밀접하게 동작 → 예외 처리가 매우 중요합니다.

- **`FileNotFoundError`**: 지정한 경로에 파일이 없을 때
  - 해결: `os.path.exists()`로 확인 또는 `try-except`로 처리
- **`PermissionError`**: 다른 프로그램이 파일을 열었거나 쓰기 권한 없음
  - 해결: 파일 사용 중 여부 확인, 로봇 시스템에서 `sudo` 권한 필요 여부 점검
- **`UnicodeDecodeError`**: 파일 인코딩(UTF-8, CP949 등)과 파이썬 설정 불일치
  - 해결: `open()` 호출 시 `encoding="utf-8"` 옵션 명시
- **`JSONDecodeError`**: JSON 문법 오류(괄호 누락 등)
  - 해결: 저장 파일 형식 확인, 빈 파일 여부 체크

### 1.8 예외 처리 — 안정적인 로봇 루프 유지

로봇 소프트웨어는 예상치 못한 상황에서도 멈추지 않아야 합니다.

In [6]:
import json
import os

def load_robot_config(path):
    if not os.path.exists(path):
        print(f"[경고] 설정 파일 없음: {path}")
        return {}

    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        print(f"[오류] JSON 형식 오류: {e}")
        return {}
    except PermissionError:
        print("[오류] 파일 접근 권한 없음")
        return {}

In [10]:
# 존재하지 않는 파일 → 경고 출력 후 빈 딕셔너리 반환
config = load_robot_config("robot_config.json")
print(f"설정 로드 결과: {config}")

[오류] 파일 접근 권한 없음
설정 로드 결과: {}


In [11]:
# robot_logs.json으로 테스트
config = load_robot_config("robot_logs.json")
print(f"robot_id: {config.get('robot_id', '없음')}")

robot_id: Alpha-01


In [13]:
sensor_data = {
    "timestamp": str(datetime.datetime.now()),
    "robot_id": "Alpha-01",
    "lidar_samples": [1.2, 1.5, 0.8, 2.3, 1.1],
    "battery_level": 85.5,
    "status": "NORMAL"
}

sensor_str = json.dumps(sensor_data)

In [14]:
import time

with open("my_robot_log.txt", "w") as f:
    for i in range(100):
        sensor_data = {
            "timestamp": str(datetime.datetime.now()),
            "robot_id": "Alpha-01",
            "lidar_samples": [1.2, 1.5, 0.8, 2.3, 1.1],
            "battery_level": 85.5,
            "status": "NORMAL"
        }

        sensor_str = json.dumps(sensor_data)
        f.write(sensor_str)
        f.write("\n")
        time.sleep(0.1)

In [17]:
with open("my_robot_log.txt", "r") as f:
    for line in f:
        data = json.loads(line)
        print(data)

{'timestamp': '2026-04-22 11:42:27.058531', 'robot_id': 'Alpha-01', 'lidar_samples': [1.2, 1.5, 0.8, 2.3, 1.1], 'battery_level': 85.5, 'status': 'NORMAL'}
{'timestamp': '2026-04-22 11:42:27.158730', 'robot_id': 'Alpha-01', 'lidar_samples': [1.2, 1.5, 0.8, 2.3, 1.1], 'battery_level': 85.5, 'status': 'NORMAL'}
{'timestamp': '2026-04-22 11:42:27.258980', 'robot_id': 'Alpha-01', 'lidar_samples': [1.2, 1.5, 0.8, 2.3, 1.1], 'battery_level': 85.5, 'status': 'NORMAL'}
{'timestamp': '2026-04-22 11:42:27.359252', 'robot_id': 'Alpha-01', 'lidar_samples': [1.2, 1.5, 0.8, 2.3, 1.1], 'battery_level': 85.5, 'status': 'NORMAL'}
{'timestamp': '2026-04-22 11:42:27.459574', 'robot_id': 'Alpha-01', 'lidar_samples': [1.2, 1.5, 0.8, 2.3, 1.1], 'battery_level': 85.5, 'status': 'NORMAL'}
{'timestamp': '2026-04-22 11:42:27.559860', 'robot_id': 'Alpha-01', 'lidar_samples': [1.2, 1.5, 0.8, 2.3, 1.1], 'battery_level': 85.5, 'status': 'NORMAL'}
{'timestamp': '2026-04-22 11:42:27.660110', 'robot_id': 'Alpha-01', 'l

In [21]:
import pickle

sensor_data = {
    "timestamp": datetime.datetime.now(),
    "robot_id": "Alpha-01",
    "lidar_samples": [1.2, 1.5, 0.8, 2.3, 1.1],
    "battery_level": 85.5,
    "status": "NORMAL"
}

with open("robot_config.pkl", "wb") as f:
    pickle.dump(sensor_data, f)

with open("robot_config.pkl", "rb") as f:
    print(pickle.load(f))

{'timestamp': datetime.datetime(2026, 4, 22, 11, 49, 40, 412206), 'robot_id': 'Alpha-01', 'lidar_samples': [1.2, 1.5, 0.8, 2.3, 1.1], 'battery_level': 85.5, 'status': 'NORMAL'}


---
## Part 2: 정규표현식 (Regex)

### 2.1 정규표현식이 왜 필요한가 — 비정형 로그 파일 파싱

자율 주행 시스템은 초당 수십 건 이상의 비정형 로그 데이터를 생성합니다.

- **성능 진단**: 수천 줄의 로그 중 특정 시간의 `ERROR` 메시지만 골라내기 — "모래사장에서 바늘 찾기" 작업을 자동화
- **레거시 센서 데이터 처리**: GPS(NMEA 문자열), 직렬(Serial) 통신 센서 등 고유 형식의 문자열을 파이썬 딕셔너리 같은 구조화된 데이터로 변환

```
[2026-04-07 10:00:12] ERROR: Motor torque limit exceeded (Value: 42.7)
[2026-04-07 10:00:15] INFO: Sensor update - Temp: 24.3, Humid: 40
```

정규표현식 없이 이 데이터를 파싱하면 → 복잡한 문자열 분리 코드 필요

### 2.2 핵심 패턴 문법 — 메타 문자

정규표현식은 메타 문자(Metacharacters)를 사용하여 특정 규칙을 정의합니다.

| 메타 문자 | 의미 |
|-----------|------|
| `.` | 줄바꿈을 제외한 **모든 한 문자**와 매치 |
| `*` | 바로 앞 문자가 **0번 이상** 반복 |
| `+` | 바로 앞 문자가 **1번 이상** 반복 |
| `?` | 바로 앞 문자가 **0번 또는 1번** (옵션) |
| `[]` | 대괄호 안의 **문자들 중 하나**와 매치 (`[0-9]`는 모든 숫자) |
| `^` / `$` | 각각 문자열의 **시작**과 **끝** |

**자주 사용하는 단축 표현(Shorthand)**

| 패턴 | 의미 | 예시 매치 |
| :--- | :--- | :--- |
| `\d` | 모든 숫자 (`[0-9]`와 동일) | `0`~`9` |
| `\d+` | 1개 이상의 숫자 | `42`, `100` |
| `\d+\.\d+` | 실수 형태 | `24.3`, `85.5` |
| `\w` | 알파벳, 숫자, 언더바(`_`) | `INFO`, `Alpha_01` |
| `\s` | 공백 문자(스페이스, 탭 등) | ` `, `\t` |
| `.*?` | 비탐욕적 임의 문자열 | 최소 매칭 |

### 2.3 주요 함수 차이와 사용법

`re` 모듈의 4가지 핵심 함수

- **`re.match()`**: 문자열의 **시작 부분**부터 패턴 일치 여부 확인
- **`re.search()`**: 문자열 **전체**를 훑으며 패턴 일치 부분 탐색
- **`re.findall()`**: 패턴과 일치하는 **모든 부분**을 리스트로 반환
- **`re.sub()`**: 일치하는 패턴을 **다른 문자열로 교체**

In [22]:
import re

log = "[ERROR] Motor overheat at node_5"

print(re.match(r"ERROR", log))          # None (시작이 '[' 이므로)
print(re.search(r"ERROR", log))         # Match! (전체에서 탐색)
print(re.findall(r"\d+", log))          # ['5']
print(re.sub(r"ERROR", "WARN", log))    # "[WARN] Motor overheat at node_5"

None
<re.Match object; span=(1, 6), match='ERROR'>
['5']
[WARN] Motor overheat at node_5


In [ ]:
text = "123asdASD가나다"

re.sub(r"[가-힣]", "*", text)

'123******가나다'

In [ ]:
re.sub(r"[0-9]", "*", text)

In [ ]:
re.sub(r"[a-zA-Z]", "*", text)

In [24]:
ord('가'), ord('힣')

(44032, 55203)

In [25]:
chr(44032), chr(55203)

('가', '힣')

In [27]:
for i in range(44032, 44032 + 100):
    print(f"{i}: {chr(i)}")

44032: 가
44033: 각
44034: 갂
44035: 갃
44036: 간
44037: 갅
44038: 갆
44039: 갇
44040: 갈
44041: 갉
44042: 갊
44043: 갋
44044: 갌
44045: 갍
44046: 갎
44047: 갏
44048: 감
44049: 갑
44050: 값
44051: 갓
44052: 갔
44053: 강
44054: 갖
44055: 갗
44056: 갘
44057: 같
44058: 갚
44059: 갛
44060: 개
44061: 객
44062: 갞
44063: 갟
44064: 갠
44065: 갡
44066: 갢
44067: 갣
44068: 갤
44069: 갥
44070: 갦
44071: 갧
44072: 갨
44073: 갩
44074: 갪
44075: 갫
44076: 갬
44077: 갭
44078: 갮
44079: 갯
44080: 갰
44081: 갱
44082: 갲
44083: 갳
44084: 갴
44085: 갵
44086: 갶
44087: 갷
44088: 갸
44089: 갹
44090: 갺
44091: 갻
44092: 갼
44093: 갽
44094: 갾
44095: 갿
44096: 걀
44097: 걁
44098: 걂
44099: 걃
44100: 걄
44101: 걅
44102: 걆
44103: 걇
44104: 걈
44105: 걉
44106: 걊
44107: 걋
44108: 걌
44109: 걍
44110: 걎
44111: 걏
44112: 걐
44113: 걑
44114: 걒
44115: 걓
44116: 걔
44117: 걕
44118: 걖
44119: 걗
44120: 걘
44121: 걙
44122: 걚
44123: 걛
44124: 걜
44125: 걝
44126: 걞
44127: 걟
44128: 걠
44129: 걡
44130: 걢
44131: 걣


### 2.4 그룹(Group)과 이름 붙은 캡처 활용법

괄호 `()`로 특정 부분을 그룹으로 묶어 나중에 따로 추출할 수 있습니다.

- **일반 그룹**: `(\d+)` → `.group(1)`로 접근
- **이름 붙은 그룹 (Named Groups)**: `(?P<name>...)` → `.group('name')`으로 접근
  - 코드가 훨씬 직관적, 로봇 로그 분석에서 특히 유용

In [28]:
import re

log = "[2026-04-07 10:00:12] ERROR: Motor torque exceeded"

In [29]:
# 이름 붙은 그룹으로 타임스탬프, 레벨, 메시지 캡처
pattern = r"\[(?P<timestamp>.*?)\] (?P<level>\w+): (?P<msg>.*)"
match = re.search(pattern, log)

if match:
    print(match.group('timestamp'))  # 2026-04-07 10:00:12
    print(match.group('level'))      # ERROR
    print(match.group('msg'))        # Motor torque exceeded

2026-04-07 10:00:12
ERROR
Motor torque exceeded


### 2.5 로봇 로그 파싱 예제

In [30]:
import re

# 실제 로봇 로그 예시 데이터
log_data = [
    "[2026-04-07 10:00:01] INFO: Battery level is 85.5%",
    "[2026-04-07 10:00:05] WARN: Lidar scan frequency dropped",
    "[2026-04-07 10:00:12] ERROR: Motor torque limit exceeded (Value: 42.7)",
    "[2026-04-07 10:00:15] INFO: Sensor update - Temp: 24.3, Humid: 40"
]

In [31]:
# 1. 타임스탬프 및 로그 레벨 추출 패턴
log_pattern = r"\[(?P<timestamp>.*?)\] (?P<level>\w+): (?P<msg>.*)"

for line in log_data:
    match = re.search(log_pattern, line)
    if match:
        # 2. 오류 레벨 필터링
        if match.group('level') == "ERROR":
            print(f"[긴급] 시간: {match.group('timestamp')}")
            print(f"       내용: {match.group('msg')}")

[긴급] 시간: 2026-04-07 10:00:12
       내용: Motor torque limit exceeded (Value: 42.7)


In [32]:
# 3. 특정 센서 값(실수) 추출
sensor_log = "Sensor update - Temp: 24.3, Humid: 40"

# 숫자.숫자 형태를 찾아 캡처
temp_match = re.search(r"Temp: (?P<temp>\d+\.\d+)", sensor_log)
if temp_match:
    print(f"현재 온도: {temp_match.group('temp')}°C")

현재 온도: 24.3°C


In [33]:
humid_match = re.search(r"Humid: (?P<humid>\d+)", sensor_log)
if humid_match:
    print(f"현재 습도: {humid_match.group('humid')}°")

현재 습도: 40°


### 2.6 입문자가 자주 하는 실수

**① `match()`와 `search()`의 혼동**
- 문자열 중간 패턴 탐색에 `match()`를 쓰면 아무것도 찾지 못함
- 항상 **`search()`를 먼저** 고려할 것

**② 역슬래시 함정 (Backslash Plague)**
- 파이썬 문자열과 Regex 모두 `\`를 특수 문자로 사용 → 충돌
- 반드시 **원시 문자열 `r"pattern"`** 표기법 사용

In [34]:
import re

# 잘못된 예: "\\d+" 처럼 이중 역슬래시 필요
print(re.search("\\d+", "42"))    # 동작하지만 가독성이 나쁨

# 올바른 예: raw string으로 간결하게
print(re.search(r"\d+", "42"))    # 권장

<re.Match object; span=(0, 2), match='42'>
<re.Match object; span=(0, 2), match='42'>


### 2.7 탐욕적 매칭(Greedy) & 문자열 메서드 선택

**탐욕적(Greedy) 매칭 주의**
- `.*`은 가능한 가장 긴 문자열을 먹어치움
- 로그에서 `[시간] 메시지` 형태 파싱 시 `.*`을 쓰면 마지막 대괄호까지 포함될 수 있음
- **`.*?`(비탐욕적, Non-greedy)** 수량자 사용 권장

**문자열 메서드로 충분한 경우**
- 단순 포함 여부: `re` 모듈보다 `in` 연산자 또는 `str.replace()`가 훨씬 빠름

In [ ]:
import re

text = "[2026-04-07] [ERROR] Motor failed"

# 탐욕적: 처음 [ 부터 마지막 ] 까지 전부 매칭
print(re.search(r"\[.*\]", text).group())    # '[2026-04-07] [ERROR]'

# 비탐욕적: 최소 범위만 매칭
print(re.search(r"\[.*?\]", text).group())   # '[2026-04-07]'

---
## Part 3: 퀴즈 · 복습 문제

### 3.1 코드 읽기 문제

아래 코드의 출력 결과를 예측하고 직접 실행하여 확인하세요.

In [35]:
# Q1. JSON 설정 로드 — 출력 결과는?
import json
config_data = '{"robot_id": "R1", "max_speed": 1.5, "sensors": ["lidar", "imu"]}'
config = json.loads(config_data)
print(config["sensors"])

['lidar', 'imu']


In [36]:
# Q2. 로그에서 패턴 추출 — 출력 리스트의 내용은?
import re
log = "SENSOR_ID: 101, TEMP: 25.4C | SENSOR_ID: 102, TEMP: 26.1C"
ids = re.findall(r"SENSOR_ID: (\d+)", log)
print(ids)

['101', '102']


In [37]:
# Q3. 파일 쓰기 모드 — 최종 파일에 몇 줄?
with open("robot_log.txt", "w") as f:
    f.write("System Start\n")
with open("robot_log.txt", "a") as f:
    f.write("Sensor Active\n")

# 파일 내용 확인
with open("robot_log.txt", "r") as f:
    print(f.read())

System Start
Sensor Active



In [38]:
# Q4. JSON 직렬화 — json_string의 자료형(type)은?
import json
status = {"battery": 85, "moving": True}
json_string = json.dumps(status, indent=4)
print(type(json_string))

<class 'str'>


In [39]:
# Q5. 정규표현식 — 출력 결과는?
import re
log_entry = "Distance: 150cm"
match = re.search(r"\d+\s?\w+", log_entry)
print(match.group() if match else "No Match")

150cm


### 3.2 코드 완성 문제

`# TODO` 부분을 채워서 코드를 완성하세요.

In [48]:
# Q1. robot_logs.json 파일을 안전하게 읽어오는 코드를 완성하세요.
import json

# TODO: 여기에 구현하세요
with open("robot_logs.json", "r") as f:
    data = json.load(f)
    print(data)

{'timestamp': '2026-04-22 11:29:49.265598', 'robot_id': 'Alpha-01', 'lidar_samples': [1.2, 1.5, 0.8, 2.3, 1.1], 'battery_level': 85.5, 'status': 'NORMAL'}


In [49]:
# Q2. 로그에서 배터리 수치를 찾는 패턴을 완성하세요.
import re
log = "Status: Battery 15%"

# TODO: 여기에 구현하세요 — 패턴의 빈칸을 채우세요
match = re.search(r"(\d+)", log)
if match:
    print(match.group(1))

15


In [50]:
# Q3. 보정 데이터를 calib.json에 저장하세요.
import json
calibration = {"offset": 0.05, "gain": 1.1}

# TODO: 여기에 구현하세요
with open("calibration.json", "w") as f:
    json.dump(calibration, f)

In [51]:
# Q4. 로그에서 대괄호 안의 날짜만 추출하는 패턴을 완성하세요.
import re
log = "[2026-04-07] INFO: Motor Active"

# TODO: 여기에 구현하세요 — 패턴의 빈칸을 채우세요
match = re.search(r"\[(.*?)\]", log)
if match:
    print(match.group(1))

2026-04-07


In [57]:
# Q5. 기존 내용을 지우지 않고 센서 데이터를 파일 끝에 추가하세요.

# TODO: 여기에 구현하세요 — 올바른 파일 열기 모드를 선택하세요
with open("sensor_log.txt", "a") as f:
    f.write("\nhello world\n")

!tail sensor_log.txt

Sensor Data: 10.5
hello world

hello world

hello world


### 3.3 오류 찾기 문제

아래 코드의 문제점을 파악하고 올바르게 수정하세요.

In [61]:
# Q1. 파일 자원 관리 누수 — 무엇이 문제인가요?
# 문제 코드:
#   f = open("sensor_data.csv", "r")
#   data = f.read()
#   # f.close()를 호출하지 않아 파일 핸들이 누수됨!

# TODO: 여기에 구현하세요 — with 문을 사용해 올바르게 수정하세요
with open("sensor_log.txt", "r") as f:
    data = f.read()


In [ ]:
# Q2. JSON 읽기 오류 — 어떤 함수를 써야 하나요?
# 문제 코드:
#   import json
#   json_str = '{"voltage": 12.0}'
#   data = json.load(json_str)  # 오류 발생! load()는 파일 객체가 필요

# TODO: 여기에 구현하세요 — 문자열에는 json.loads()를 사용해야 합니다
json_str = '{"voltage" : 12.0}'
data = json.loads(json_str)

In [ ]:
# Q3. 보안 위험 — 아래 코드에서 왜 위험한가요?
# 문제 코드:
#   import json
#   raw_data = '{"cmd": "reboot"}'
#   data = eval(raw_data)  # eval()은 신뢰할 수 없는 원격 데이터를 실행할 위험!

# TODO: 여기에 구현하세요 — eval() 대신 json.loads()로 안전하게 파싱하세요
raw_data = '{"cmd" : "reboot"}'
data = json.loads(raw_data)

In [64]:
eval("print('hello world')")

exec("""for i in range(10):
        print(i)""")

hello world
0
1
2
3
4
5
6
7
8
9
